# Validação Full-Corpus — Recall no Manga109 inteiro

Notebook de **inferência pura** (sem treino): carrega os pesos já commitados no repositório (`weights/best.pt` detector, `weights/classifier_best.pt` classificador) e mede o recall do pipeline completo em todo o corpus Manga109 (109 volumes, ~10.600 páginas), complementando o benchmark de 4 páginas curadas manualmente (`src/helper/benchmark.py`) com uma medida em escala.

**Checklist antes de rodar:**
1. Anexe o dataset do Manga109 completo (imagens + anotações) no painel lateral direito em **+ Add Input → Datasets**.
2. Habilite a GPU (opcional, mas acelera): *Session options → Accelerator → GPU T4 x2 ou P100*.
3. **Run All**.


In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "GPU nao encontrada (ok, roda em CPU tambem -- so mais lento).")


## 1. Parâmetros do experimento

In [ ]:
# Numero de volumes pro dry-run inicial (mede tempo antes de comprometer ao
# corpus inteiro). Depois de conferir o tempo extrapolado, rode a celula do
# run completo (secao 6) separadamente.
DRY_RUN_LIMIT_VOLUMES = 3


## 2. Instalar dependências

In [ ]:
!pip install -q ultralytics
print("Dependencias instaladas.")


## 3. Configurar repositório

Clona (ou atualiza) o repositório em `/kaggle/working/`. `weights/best.pt` e `weights/classifier_best.pt` já vêm commitados no clone -- não precisa anexar dataset separado pros pesos.

In [ ]:
import os
import sys
import shutil

WORK_DIR  = "/kaggle/working"
REPO_NAME = "Detector-de-kanjis-n1"
REPO_DIR  = os.path.join(WORK_DIR, REPO_NAME)
REPO_URL  = f"https://github.com/MiguelMussalam/{REPO_NAME}.git"

is_valid_repo = os.path.isdir(os.path.join(REPO_DIR, ".git"))

if not is_valid_repo:
    if os.path.exists(REPO_DIR):
        print(f"Diretorio {REPO_DIR} existe mas nao e um repo git valido. Removendo...")
        shutil.rmtree(REPO_DIR)
    print(f"Clonando {REPO_URL} ...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo valido encontrado. Atualizando...")
    !git -C {REPO_DIR} pull

assert os.path.isfile(os.path.join(REPO_DIR, "config.py")), \
    f"config.py nao encontrado em {REPO_DIR} -- verifique se o clone funcionou"

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from config import DETECTOR_WEIGHTS_PATH, CLF_WEIGHTS_PATH, MANGA109_IMAGES, MANGA109_ANNOTATIONS

assert os.path.exists(DETECTOR_WEIGHTS_PATH), f"Peso do detector nao encontrado: {DETECTOR_WEIGHTS_PATH}"
assert os.path.exists(CLF_WEIGHTS_PATH), f"Peso do classificador nao encontrado: {CLF_WEIGHTS_PATH}"
assert os.path.isdir(MANGA109_IMAGES) and os.path.isdir(MANGA109_ANNOTATIONS), \
    "Manga109 nao encontrado -- confira se o dataset foi anexado (+ Add Input)."

print(f"Diretorio de trabalho: {os.getcwd()}")
print(f"Detector:     {DETECTOR_WEIGHTS_PATH}")
print(f"Classificador: {CLF_WEIGHTS_PATH}")
print(f"Manga109:     {MANGA109_IMAGES}")


## 4. Construir ground truth full-corpus

Percorre todo XML do Manga109 (bbox de linha `<text>` + transcrição oficial), sem rodar nenhum modelo -- rápido (segundos, não minutos). Aplica o filtro heurístico automático de transcrição suspeita (ver `src/helper/manga109_corpus.py`) no lugar da verificação visual manual (inviável em ~10.600 páginas).

In [ ]:
!python -m src.helper.manga109_corpus


## 5. Dry-run — medir tempo antes do run completo

`Pipeline.predict()` faz um forward do classificador por detecção (sem batching) -- uma página de manga pode ter 100-300+ caracteres. Mede tempo/página numa amostra pequena e extrapola pro corpus inteiro antes de comprometer ao run completo.

In [ ]:
!python -m src.helper.corpus_validate --limit-volumes {DRY_RUN_LIMIT_VOLUMES} --out /kaggle/working/resultado_dryrun.json


In [ ]:
import json

with open("/kaggle/working/resultado_dryrun.json", encoding="utf-8") as f:
    dry = json.load(f)

print(f"{dry['n_paginas']} paginas em {dry['duracao_segundos']:.1f}s "
      f"({dry['segundos_por_pagina']:.3f}s/pagina)")
print(f"Extrapolado pro corpus completo (~10602 paginas): "
      f"{dry['extrapolado_corpus_completo_horas']:.2f}h")
print(f"Recall nessa amostra: {dry['recall_pct']:.1f}%  |  OUTROS: {dry['outros_pct']:.1f}%")


## 6. Confirmação antes do run completo

Confira o tempo extrapolado acima. Se couber no orçamento de sessão do Kaggle (~9h GPU), prossiga para a próxima célula. Se não couber, considere `--sample-paginas-por-volume K` em vez do corpus inteiro (amostra grande, não exaustiva).

## 7. Run completo

In [ ]:
!python -m src.helper.corpus_validate --out /kaggle/working/resultado_full.json


## 8. Resultados

In [ ]:
import pandas as pd

with open("/kaggle/working/resultado_full.json", encoding="utf-8") as f:
    resultado = json.load(f)

print(f"Paginas avaliadas: {resultado['n_paginas']}")
print(f"Deteccoes totais:  {resultado['deteccoes_total']} ({resultado['outros_pct']:.1f}% OUTROS)")
print(f"Recall agregado:   {resultado['hits_total']}/{resultado['esperado_total']} "
      f"({resultado['recall_pct']:.1f}%)")

df_volumes = pd.DataFrame([
    {"volume": vol, **stats}
    for vol, stats in resultado["por_volume"].items()
]).sort_values("recall_pct")

df_volumes


## 9. Compactar e baixar

In [ ]:
import zipfile
from IPython.display import FileLink, display

zip_name = "/kaggle/working/resultados_validacao_corpus.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for fname in ["resultado_full.json", "resultado_dryrun.json"]:
        fpath = f"/kaggle/working/{fname}"
        if os.path.exists(fpath):
            zipf.write(fpath, fname)

    gt_path = os.path.join(REPO_DIR, "data", "corpus_validation", "ground_truth_full.json")
    if os.path.exists(gt_path):
        zipf.write(gt_path, "ground_truth_full.json")

print(f"Zip criado: {zip_name}")
display(FileLink(zip_name))
